In [13]:
from src.post_processing import PathWrangler
import polars as pl
from pathlib import Path
from collections import defaultdict
from ergochemics.draw import draw_reaction, draw_molecule
from ergochemics.mapping import get_reaction_center
from ergochemics.similarity import MolFeaturizer, ReactionFingerprinter
from rdkit import Chem
from IPython.display import display, SVG
import numpy as np
from hydra import initialize, compose

In [14]:
# study = "/home/stef/quest_data/bottle/data/processed/260327_target_3hpa"
study = "/home/stef/quest_data/bottle/data/processed/260410_target_3hpa"
# study = "/projects/b1039/spn1560/bottle/data/processed/260327_target_3hpa"
# study = "/home/stef/quest_data/bottle/data/processed/260324_2_step_bi_akg_to_hopa"
known = "/home/stef/bottle/artifacts/known"
# out_dir = "/home/stef/bottle/artifacts/coa_mutase_paths"

Direct look at parquet files

In [15]:
with initialize(config_path="../conf/filepaths", version_base=None):
    filepaths = compose(config_name="filepaths")

In [16]:
cpds = pl.read_parquet(
    Path(study)  / "compounds.parquet"
)
cpds.head()

id,smiles,type,name
str,str,enum,str
"""07e48b419b37606b338ba3e838a8e5…","""Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O…","""helper""","""PYROPHOSPHATE_DONOR_CoF"""
"""4ea34a41c2091b9cd312119934909d…","""Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O…","""helper""","""PYROPHOSPHATE_ACCEPTOR_CoF"""
"""c1a94434194999200be197465c01c3…","""Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(C…","""helper""","""FAD_CoF"""
"""37225a49e3ed17e65590fb5ed4d536…","""Cc1cc2c(cc1C)N(CC(O)C(O)C(O)CO…","""helper""","""FADH2_CoF"""
"""d149421ac53cc62325e84c1688210e…","""Nc1ncnc2c1ncn2C1OC(COP(=O)(O)O…","""helper""","""PHOSPHATE_ACCEPTOR_CoF"""


In [17]:
paths = pl.read_parquet(
    Path(study) / "paths.parquet"
)
paths.head()

path_id,rxn_id,main_pdt_id,rxn_type,generation
str,str,str,enum,i32
"""0bc7301a6d8122e4ae16c4cfe5eb63…","""3238d364a7911ad3440c41c05c944a…","""cd431e4c6b054540791647925df16a…","""predicted""",1
"""0ccd84b14206a2bec2dbdf83ee4e90…","""efb1c1d2bd82a3e0bb262e80001631…","""692a548c92224b659ae0c86b41bc9c…","""predicted""",1
"""cf7dff1b59c7b6e8099361663166aa…","""f4028197d2316361e513527282ccca…","""44a245a4a8752c17f6ab3175f1aed4…","""predicted""",0
"""f9cc32cb062e5fc84f84a6d0a68e68…","""e00d2fd6f101dd3d9ac4cda139cf0c…","""7adf4b7cf3272a4091055af7ffa13e…","""predicted""",2
"""860938b6f1f9e3d6de05bccc90bb49…","""469d673e0020a199723ebddef56551…","""6569ea8ebe9fdd8627adb76f7d942d…","""predicted""",4


In [18]:
path_stats = pl.read_parquet(
    Path(study) / "path_stats.parquet"
)
path_stats.head()

id,starters,targets,dg_opt,dg_err,starter_ids,target_ids,mdf,mean_max_rxn_sim,mean_mean_rxn_sim,min_max_rxn_sim,min_mean_rxn_sim,feasibility_frac
str,list[str],list[str],list[f32],list[f32],list[str],list[str],f32,f32,f32,f32,f32,f32
"""5e1957b85a15837c20463a3b3cd150…","[""serine""]","[""3hpa""]","[-1724.078491, 1845.854736, … -920.131775]","[-122474.484375, 0.000004, … 0.000014]","[""26d1da597f251b7677489fe81ff93b9675c82fb2""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",-1845.854736,0.469438,0.140728,0.285714,0.071105,0.4
"""40c09c78c51f9d628218d678aa133a…","[""serine""]","[""3hpa""]","[-1714.969727, -4.07868, … -933.512207]","[-1.4898e-11, -2.1132e-12, … -89442.71875]","[""26d1da597f251b7677489fe81ff93b9675c82fb2""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",4.07868,0.526352,0.222681,0.163043,0.05259,0.4
"""c23194575ab132eec483fc20b405de…","[""serine""]","[""3hpa""]","[-1040.783813, -18.191639, … 122.474617]","[-141421.359375, 0.000004, … 0.000024]","[""26d1da597f251b7677489fe81ff93b9675c82fb2""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",-122.474617,0.26627,0.086288,0.167019,0.038517,0.4
"""9a2aab407e7b2d42776817af80f697…","[""succinate""]","[""3hpa""]","[-22.719717, -23.63415, … -29.316637]","[-0.000005, 0.000007, … 8.7741e-8]","[""fae11404b12230ded37cfd2511bf3dd98bdacd3c""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",19.888083,0.637732,0.368262,0.111732,0.094393,0.6
"""1b7f047a63f43bbd19e850a7fdae38…","[""ketoglutarate""]","[""3hpa""]","[-7.058908, -7.058908, … -10.177712]","[0.000005, -0.000003, … -9.6620e-7]","[""6cf6db1d70f43061de31deeb417214c6fd7ad005""]","[""6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d""]",7.058908,0.73659,0.533775,0.168831,0.060807,0.6


In [19]:
print(paths.shape)
print(paths.unique().shape)
print(path_stats.shape)
print(path_stats.unique().shape)

(115261, 5)
(115261, 5)
(23400, 13)
(23400, 13)


In [20]:
pred_rxns = pl.read_parquet(
    Path(study) / "predicted_reactions.parquet"
)
print(pred_rxns.shape)
pred_rxns.head()

(7128, 9)


id,smarts,am_smarts,dxgb_label,rxn_sims,analogue_ids,rules,templates,rule_sets
str,str,str,i32,list[f32],list[str],list[str],list[str],list[str]
"""b01ec78f2d79e43d56933b4ce302ff…","""CCCO.O=C=O>>O=C(O)CCCO""","""[CH3:4][CH2:5][CH2:6][OH:7].[O…",0,"[0.306122, 0.306122, … 0.004098]","[""840781944e689e93343dc2a068ba227ff61a0d2b"", ""1c3e7c7c48c53614c75a647942e6da0b3f3d5309"", … ""76d9070c1efc53db1a2a36c4f1fef1f3343e0a0f""]","[""499_0_reversed""]","[""[C&D1&v4&H3&+0&!R&z0:1].[O&D1&v2&H0&+0&!R:3]=[C&D2&v4&H0&+0&!R&z2:2]>>[C&D2&v4&H2&+0&!R&z0:1]-[C&D3&v4&H0&+0&!R&z2:2]-[O&D1&v2&H1&+0&!R:3]""]","[""mechinferred_dt_224_rules_w_coreactants""]"
"""5d05fb0be54ef57ed44932ac1660b5…","""O=C(O)CCC(=O)O.NC(COP(=O)(O)OP…","""[O:1]=[C:2]([OH:3])[CH2:4][CH2…",1,"[0.183406, 0.171779, … 0.084317]","[""f843c12be8a52cc29ff38de4e5de1b0aecf7feb4"", ""973959253d721c528f93df2b99af7c32c0dc10ef"", … ""b418bc48915849fbd8a66e819ed3825f97feb2a0""]","[""143_1_0_0"", ""143_1"", ""143_1_0""]","[""[C&D3&v4&H0&+0&!R&z2:1](=[O&D1&v2&H0&+0&!R:2])-[O&D1&v2&H1&+0&!R:3].[P&D4&v5&H0&+0&!R:4]-[O&D2&v2&H0&+0&!R:5]-[P&D4&v5&H0&+0&!R:6].[S&D1&v2&H1&+0&!R:7]>>[C&D3&v4&H0&+0&!R&z2:1](=[O&D1&v2&H0&+0&!R:2])-[S&D2&v2&H0&+0&!R:7].[P&D4&v5&H0&+0&!R:6]-[O&D1&v2&H1&+0&!R:5].[P&D4&v5&H0&+0&!R:4]-[O&D1&v2&H1&+0&!R:3]"", ""[C&D3&v4&H0&+0&!R&z2:1](=[O&D1&v2&H0&+0&!R:2])-[O&D1&v2&H1&+0&!R:3].[P&D4&v5&H0&+0&!R:4]-[O&D2&v2&H0&+0&!R:5]-[P&D4&v5&H0&+0&!R:6].[S&D1&v2&H1&+0&!R:7]>>[C&D3&v4&H0&+0&!R&z2:1](=[O&D1&v2&H0&+0&!R:2])-[S&D2&v2&H0&+0&!R:7].[P&D4&v5&H0&+0&!R:6]-[O&D1&v2&H1&+0&!R:5].[P&D4&v5&H0&+0&!R:4]-[O&D1&v2&H1&+0&!R:3]"", ""[C&D3&v4&H0&+0&!R&z2:1](=[O&D1&v2&H0&+0&!R:2])-[O&D1&v2&H1&+0&!R:3].[P&D4&v5&H0&+0&!R:4]-[O&D2&v2&H0&+0&!R:5]-[P&D4&v5&H0&+0&!R:6].[S&D1&v2&H1&+0&!R:7]>>[C&D3&v4&H0&+0&!R&z2:1](=[O&D1&v2&H0&+0&!R:2])-[S&D2&v2&H0&+0&!R:7].[P&D4&v5&H0&+0&!R:6]-[O&D1&v2&H1&+0&!R:5].[P&D4&v5&H0&+0&!R:4]-[O&D1&v2&H1&+0&!R:3]""]","[""mechinferred_dt_956_rules_w_coreactants"", ""mechinferred_dt_956_rules_w_coreactants"", ""mechinferred_dt_956_rules_w_coreactants""]"
"""ec015573a8794d1fccd2047550194c…","""O=C(O)CC(=O)C(=O)O.O=C=O.CC(C)…","""[O:1]=[C:2]([OH:3])[CH2:4][C:5…",0,"[0.810256, 0.810256, … 0.076289]","[""95c37747aef287e1800c13465df0ed0c7fa88f95"", ""0f0d3257dfae9e15c7ab31fc04ea5dca269127eb"", … ""2af7a70272691fa256d1bcf12e0742d5166ca9b5""]","[""434_2""]","[""[C&D2&v4&H2&+0&!R&z0:1]-[C&D3&v4&H0&+0&!R&z1:2]=[O&D1&v2&H0&+0&!R:3].[O&D1&v2&H0&+0&!R:4]=[C&D2&v4&H0&+0&!R&z2:5].[S&D1&v2&H1&+0&!R:6]>>[C&D2&v4&H2&+0&!R&z0:1]-[C&D3&v4&H0&+0&!R&z2:5]-[O&D1&v2&H1&+0&!R:4].[S&D2&v2&H0&+0&!R:6]-[C&D3&v4&H0&+0&!R&z2:2]=[O&D1&v2&H0&+0&!R:3]""]","[""mechinferred_dt_956_rules_w_coreactants""]"
"""4c27c84e8e81f88121448a827bc954…","""O=C(O)CCC(=O)OC(=O)CCC(=O)O.CC…","""[O:1]=[C:2]([OH:3])[CH2:4][CH2…",1,"[0.878866, 0.878866, … 0.025882]","[""73a3231dbf03c7a1349c4ef5311e24df4bc60a9f"", ""ce21d0f66b0f99b9ec10ff1773a8f980085db08e"", … ""de10753859371859682ec7b7d8cc2706820ee700""]","[""47_3""]","[""[C&D3&v4&H0&+0&!R&z2:1](=[O&D1&v2&H0&+0&!R:2])-[O&D2&v2&H0&+0&!R:3].[S&D1&v2&H1&+0&!R:4]>>[O&D1&v2&H1&+0&!R:3].[S&D2&v2&H0&+0&!R:4]-[C&D3&v4&H0&+0&!R&z2:1]=[O&D1&v2&H0&+0&!R:2]""]","[""mechinferred_dt_956_rules_w_coreactants""]"
"""b9ee2738571dd494392e7ea60acc8b…","""CC(C)(COP(=O)(O)OP(=O)(O)OCC1O…","""[CH3:26][C:25]([CH3:27])([CH2:…",1,"[0.845109, 0.845109, … 0.014851]","[""b4c25c288c30a8a0c671f4f171296e4f6785c7ba"", ""7bb2582ddc5f3bc4bc91a9307aea868b819e63f1"", … ""a656fb808fd69ca696d753b2fbf2328c6ebfc93d""]","[""1964_0_reversed""]","[""[C&D3&v4&H1&+0&!R&z0:2]-[C&D2&v4&H2&+0&!R&z0:1]-[C&D3&v4&H0&+0&!R&z2:3]=[O&D1&v2&H0&+0&!R:4]>>[C&D1&v4&H3&+0&!R&z0:1]-[C&D4&v4&H0&+0&!R&z0:2]-[C&D3&v4&H0&+0&!R&z2:3]=[O&D1&v2&H0&+0&!R:4]""]","[""mechinferred_dt_224_rules_w_coreactants""]"


In [21]:
path_fb = pl.read_parquet(
    Path(study) / "path_feedback.parquet"
)
path_fb.head()

username,id,feedback,date,time
str,str,i32,date,time
"""Stefan""","""679ce8c6e93c5c6e2f1d54772b57ee…",0,2026-04-17,10:51:12.290072
"""Stefan""","""0b7bb08c699db5934fb346b2b62807…",0,2026-04-17,10:51:12.290072
"""Stefan""","""dfad596a72a8de6dc5321e895bc07f…",0,2026-04-17,10:51:12.290072
"""Stefan""","""787c56aee7410d3a28b4b7b22a5536…",0,2026-04-17,10:51:12.290072
"""Stefan""","""2f65915a529f9a86773e687909be2f…",1,2026-04-17,10:51:12.290072


In [22]:
path_fb.shape

(44, 5)

In [23]:
rxn_fb = pl.read_parquet(
    Path(study) / "reaction_feedback.parquet"
)
rxn_fb.head()

username,id,feedback,date,time
str,str,i32,date,time
"""Stefan""","""5959dace01dac3a519f9d50bb3af57…",0,2026-04-16,16:15:55.825690
"""Stefan""","""fb4e2a8027e11dbec27fe6ca29a6bc…",0,2026-04-16,16:15:55.825690
"""Stefan""","""dd322af258382cbc5680502046c37b…",0,2026-04-16,16:15:55.825690
"""Stefan""","""9038f7a31b67d543ce064375f4050b…",0,2026-04-16,16:15:55.825690
"""Stefan""","""cecd6de9246a45d052a60c0829fe77…",0,2026-04-16,16:15:55.825690


In [26]:
path_fb_update = path_fb.with_columns(
    pl.col("username").map_elements(lambda x: "stefanpate94@gmail.com" if x == "Stefan" else x).alias("username")
)
path_fb_update.head()

username,id,feedback,date,time
str,str,i32,date,time
"""stefanpate94@gmail.com""","""679ce8c6e93c5c6e2f1d54772b57ee…",0,2026-04-17,10:51:12.290072
"""stefanpate94@gmail.com""","""0b7bb08c699db5934fb346b2b62807…",0,2026-04-17,10:51:12.290072
"""stefanpate94@gmail.com""","""dfad596a72a8de6dc5321e895bc07f…",0,2026-04-17,10:51:12.290072
"""stefanpate94@gmail.com""","""787c56aee7410d3a28b4b7b22a5536…",0,2026-04-17,10:51:12.290072
"""stefanpate94@gmail.com""","""2f65915a529f9a86773e687909be2f…",1,2026-04-17,10:51:12.290072


In [27]:
rxn_fb_update = rxn_fb.with_columns(
    pl.col("username").map_elements(lambda x: "stefanpate94@gmail.com" if x == "Stefan" else x).alias("username")
)
rxn_fb_update.head()

username,id,feedback,date,time
str,str,i32,date,time
"""stefanpate94@gmail.com""","""5959dace01dac3a519f9d50bb3af57…",0,2026-04-16,16:15:55.825690
"""stefanpate94@gmail.com""","""fb4e2a8027e11dbec27fe6ca29a6bc…",0,2026-04-16,16:15:55.825690
"""stefanpate94@gmail.com""","""dd322af258382cbc5680502046c37b…",0,2026-04-16,16:15:55.825690
"""stefanpate94@gmail.com""","""9038f7a31b67d543ce064375f4050b…",0,2026-04-16,16:15:55.825690
"""stefanpate94@gmail.com""","""cecd6de9246a45d052a60c0829fe77…",0,2026-04-16,16:15:55.825690
